# Destabilizing Mutation → Protein Abundance Analysis

Tests whether destabilizing missense mutations reduce protein abundance in MS data.

**Logic:**
For each destabilizing mutation (high AlphaMissense score OR low SPURS ddG) in gene X:
1. Find samples (TMT channels) that carry the mutation
2. Find samples in the same plex that do NOT carry the mutation
3. Compare shared peptide intensities of protein X between the two groups
4. A significant decrease in mutant-carrier samples = evidence of destabilization

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP      = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META     = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
MISSENSE     = "/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv"
RESULTS_BASE = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/results"
PLEX_LIST    = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
OUT_DIR      = "/scratch/leduc.an/AAS_Evo/ANALYSIS/destabilization"

# Thresholds for "destabilizing"
AM_THRESHOLD   = 0.564   # AlphaMissense likely_pathogenic cutoff
SPURS_THRESHOLD = 0.5    # ddG > 0.5 = destabilizing (positive = less stable)
VAF_THRESHOLD  = 0.3     # minimum VAF to call mutation present
N_PLEXES       = None    # None = all plexes; set to int to limit for testing

TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
    "tmt_126c":"126C","tmt_127n":"127N", "tmt_134n":"134N",  # TMTpro extras
}

os.makedirs(OUT_DIR, exist_ok=True)
print("Config loaded")

In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
if N_PLEXES:
    plex_ids = plex_ids[:N_PLEXES]

print(f"Plexes: {len(plex_ids)}")
print(f"TMT map: {len(tmt):,} rows")
print(f"GDC meta: {len(gdc):,} rows")

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print("Loading missense table...")
missense = pd.read_csv(MISSENSE, sep="\t", low_memory=False)
print(f"Total rows: {len(missense):,}")

# Filter for destabilizing mutations
am_destab  = missense["am_pathogenicity"].astype(float, errors="ignore") >= AM_THRESHOLD
spurs_destab = pd.to_numeric(missense["spurs_ddg"], errors="coerce") >= SPURS_THRESHOLD
vaf_ok     = pd.to_numeric(missense["VAF"], errors="coerce") >= VAF_THRESHOLD
gnomad_ok  = pd.to_numeric(missense["gnomADe_AF"], errors="coerce").fillna(0) < 0.01

destab = missense[(am_destab | spurs_destab) & vaf_ok & gnomad_ok].copy()
print(f"Destabilizing mutations (AM>={AM_THRESHOLD} OR SPURS>={SPURS_THRESHOLD}, VAF>={VAF_THRESHOLD}, gnomAD<0.01): {len(destab):,}")
print(f"Unique genes: {destab['SYMBOL'].nunique()}")
print(f"Unique samples: {destab['sample_id'].nunique()}")

# Build lookup: sample_id (GDC UUID) → set of mutant gene symbols
sample_mut_genes = destab.groupby("sample_id")["SYMBOL"].apply(set).to_dict()
print(f"\nSample→gene map built for {len(sample_mut_genes):,} samples")

In [ ]:
# ── HELPER: map plex TMT channels → GDC UUIDs ─────────────────────────────────
def get_channel_uuid_map(plex_id):
    """Returns dict: channel → gdc_file_id for one plex."""
    plex_tmt = tmt[tmt["run_metadata_id"] == plex_id][["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates()
    plex_tmt = plex_tmt[~plex_tmt["case_submitter_id"].str.lower().isin(["ref","reference","pooled","pool","nan"])]
    plex_tmt["channel"] = plex_tmt["tmt_channel"].map(TMT_CHANNEL_MAP)
    plex_tmt = plex_tmt.dropna(subset=["channel"])

    merged = plex_tmt.merge(
        gdc[["gdc_file_id","case_submitter_id","sample_type"]],
        on=["case_submitter_id","sample_type"], how="left"
    )
    # One channel → one UUID (take first if multiple)
    ch_uuid = merged.dropna(subset=["gdc_file_id"]).drop_duplicates("channel").set_index("channel")["gdc_file_id"].to_dict()
    return ch_uuid

print("Helper defined")

In [ ]:
# ── MAIN LOOP: collect per-gene abundance comparisons across all plexes ────────
# For each plex, load abundance_protein_MD.tsv (log2 normalized per-channel protein abundances)
# For each gene with a destabilizing mutation in that plex:
#   - mutant channels: those whose UUID has a destabilizing mutation in this gene
#   - wt channels: all other channels in the plex
#   - record the protein row's intensities for both groups

records = []  # list of dicts
skipped = []

for plex_id in plex_ids:
    results_dir = os.path.join(RESULTS_BASE, plex_id)

    # Find abundance_protein_MD.tsv (inside experiment subdirectory)
    abund_files = glob.glob(os.path.join(results_dir, "**/abundance_protein_MD.tsv"), recursive=True)
    if not abund_files:
        # fallback: tmt-report
        abund_files = glob.glob(os.path.join(results_dir, "tmt-report/abundance_protein_MD.tsv"))
    if not abund_files:
        skipped.append(plex_id)
        continue

    abund = pd.read_csv(abund_files[0], sep="\t", index_col=0)

    # Channel columns — numeric columns that aren't annotation
    annot_cols = {"NumberPSM","Protein","Gene","ProteinID","Entry","MaxPepProb","ReferenceIntensity"}
    ch_cols = [c for c in abund.columns if c not in annot_cols and abund[c].dtype in [float, "float64"]]

    # Map channels in this abundance file → GDC UUIDs
    ch_uuid = get_channel_uuid_map(plex_id)

    # Find gene column
    gene_col = next((c for c in abund.columns if c.lower() in ["gene","genename","symbol"]), None)

    for ch in ch_cols:
        uuid = ch_uuid.get(ch)
        if not uuid or uuid not in sample_mut_genes:
            continue

        mut_genes = sample_mut_genes[uuid]

        # Channels without this mutation in this plex = comparison group
        wt_chs = [c for c in ch_cols if c != ch and
                  (ch_uuid.get(c) is None or
                   gene not in sample_mut_genes.get(ch_uuid.get(c), set()))
                  for gene in mut_genes]

        for gene in mut_genes:
            # Find this gene's row in abundance table
            if gene_col:
                gene_rows = abund[abund[gene_col] == gene]
            else:
                gene_rows = abund[abund.index.str.contains(f"GN={gene} ", na=False)]
            if gene_rows.empty:
                continue

            mut_val = gene_rows.iloc[0][ch]
            # WT channels = those not carrying a destabilizing mutation in this gene
            wt_chs_gene = [c for c in ch_cols if c != ch and
                           gene not in sample_mut_genes.get(ch_uuid.get(c, ""), set())]

            if pd.isna(mut_val) or not wt_chs_gene:
                continue

            wt_vals = gene_rows.iloc[0][wt_chs_gene].dropna().values
            if len(wt_vals) < 2:
                continue

            records.append({
                "plex_id":   plex_id,
                "gene":      gene,
                "mut_uuid":  uuid,
                "channel":   ch,
                "mut_abund": mut_val,
                "wt_mean":   np.mean(wt_vals),
                "wt_std":    np.std(wt_vals),
                "wt_n":      len(wt_vals),
                "delta":     mut_val - np.mean(wt_vals),  # negative = less abundant
            })

print(f"Records collected: {len(records):,}")
print(f"Plexes skipped (no abundance file): {len(skipped)}")
df = pd.DataFrame(records)
df.head()

In [ ]:
# ── AGGREGATE PER GENE ────────────────────────────────────────────────────────
# For each gene, pool all mutant vs WT observations across plexes
# Run a one-sample t-test: is the mean delta significantly < 0?

gene_stats = []
for gene, grp in df.groupby("gene"):
    deltas = grp["delta"].values
    n = len(deltas)
    if n < 2:
        continue
    t, p = stats.ttest_1samp(deltas, popmean=0)
    gene_stats.append({
        "gene":       gene,
        "n_obs":      n,
        "mean_delta": np.mean(deltas),
        "std_delta":  np.std(deltas),
        "t_stat":     t,
        "p_value":    p,
        "n_plexes":   grp["plex_id"].nunique(),
    })

gene_stats = pd.DataFrame(gene_stats).sort_values("p_value")

# FDR correction (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
_, gene_stats["fdr"], _, _ = multipletests(gene_stats["p_value"], method="fdr_bh")

print(f"Genes tested: {len(gene_stats)}")
print(f"Significant (FDR<0.05, delta<0): {((gene_stats['fdr']<0.05) & (gene_stats['mean_delta']<0)).sum()}")
gene_stats.head(20)

In [ ]:
# ── PLOT: volcano ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

colors = gene_stats.apply(
    lambda r: "#d62728" if r["fdr"] < 0.05 and r["mean_delta"] < 0
              else ("#1f77b4" if r["fdr"] < 0.05 else "#aaaaaa"), axis=1
)

ax.scatter(gene_stats["mean_delta"], -np.log10(gene_stats["p_value"]),
           c=colors, s=20, alpha=0.7, linewidths=0)

# Label top hits
top = gene_stats[(gene_stats["fdr"] < 0.05) & (gene_stats["mean_delta"] < 0)].head(15)
for _, row in top.iterrows():
    ax.annotate(row["gene"],
                xy=(row["mean_delta"], -np.log10(row["p_value"])),
                fontsize=7, ha="left", va="bottom")

ax.axvline(0, color="k", lw=0.5, ls="--")
ax.axhline(-np.log10(0.05), color="gray", lw=0.5, ls=":")
ax.set_xlabel("Mean log2 abundance delta (mutant − WT)")
ax.set_ylabel("-log10(p-value)")
ax.set_title("Destabilizing mutations vs protein abundance")

patches = [
    mpatches.Patch(color="#d62728", label="FDR<0.05, less abundant"),
    mpatches.Patch(color="#1f77b4", label="FDR<0.05, more abundant"),
    mpatches.Patch(color="#aaaaaa", label="NS"),
]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "destab_volcano.pdf"), dpi=150)
plt.show()

In [ ]:
# ── PLOT: distribution of deltas ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["delta"], bins=80, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(0, color="k", lw=1, ls="--")
ax.axvline(df["delta"].mean(), color="red", lw=1.5, ls="-", label=f"mean={df['delta'].mean():.3f}")
ax.set_xlabel("log2 abundance delta (mutant channel − plex WT mean)")
ax.set_ylabel("Count")
ax.set_title("Distribution of abundance deltas for destabilizing mutations")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "destab_delta_dist.pdf"), dpi=150)
plt.show()

# Global test: is the mean delta < 0?
t, p = stats.ttest_1samp(df["delta"].dropna(), popmean=0)
print(f"Global one-sample t-test (delta vs 0): t={t:.3f}, p={p:.2e}")
print(f"Mean delta: {df['delta'].mean():.4f}  (negative = destabilizing mutations have lower abundance)")

In [ ]:
# ── SAVE RESULTS ──────────────────────────────────────────────────────────────
gene_stats.to_csv(os.path.join(OUT_DIR, "gene_destab_stats.tsv"), sep="\t", index=False)
df.to_csv(os.path.join(OUT_DIR, "per_observation.tsv"), sep="\t", index=False)
print(f"Saved to {OUT_DIR}")